# EoMT trên BTXRD: tham số gốc và tham số điều chỉnh theo dữ liệu (chạy local trên máy có GPU)

Bản local của `btxrd_eomt_merged_colab.ipynb`: cùng hai cấu hình `baseline` / `tuned` và cùng các bước huấn luyện, đánh giá, so sánh; chỉ khác môi trường, đường dẫn và nguồn dữ liệu.

**Chạy:**
1. Máy cần GPU NVIDIA có driver (lệnh `nvidia-smi` chạy được), Git, Python 3.10–3.13 và Jupyter (VS Code hoặc `pip install notebook`). Nên tạo môi trường riêng, ví dụ `python -m venv .venv` hoặc `conda create -n eomt python=3.12`.
2. Lấy mã nguồn: `git clone https://github.com/67htb5vn/eomt.git`, rồi mở notebook này **từ thư mục `eomt` vừa clone** (thư mục làm việc phải là thư mục repo), chọn kernel của môi trường vừa tạo, *Run All*.
3. Mục 2 tự cài PyTorch bản CUDA (`torch==2.7.0`, `torchvision==0.22.0`, như `requirements.txt`) nếu môi trường chưa có, rồi cài `requirements_btxrd.txt`. Driver cũ không hỗ trợ CUDA 12.8 thì đổi `TORCH_INDEX` sang `cu126` hoặc `cu118`.
4. Dữ liệu: mục 3 tải dataset Kaggle [`nguyetanhmai/btxrd-full-split`](https://www.kaggle.com/datasets/nguyetanhmai/btxrd-full-split). Dataset private thì cần token Kaggle: đặt file `kaggle.json` (Kaggle → *Settings* → *Create New Token*) vào `~/.kaggle/` (Windows: `C:\Users\<tên>\.kaggle\`), hoặc đặt biến môi trường `KAGGLE_USERNAME`, `KAGGLE_KEY` trước khi mở Jupyter. Đã có sẵn thư mục `BTXRD_full_split` (hoặc `BTXRD_full_split.zip`) thì chép vào trong thư mục `eomt` hoặc cạnh nó, notebook dùng bản đó và không tải nữa.
5. Lần chạy đầu cần Internet (cài thư viện, tải dữ liệu, tải checkpoint COCO từ Hugging Face); sau đó chạy offline được.
6. Bị ngắt giữa chừng thì *Run All* lại: cấu hình nào huấn luyện xong rồi thì bỏ qua, cấu hình đang dở thì huấn luyện tiếp từ `last.ckpt` (trạng thái early stopping cũng được khôi phục). Muốn chỉ chạy một cấu hình thì sửa `RUN_CONFIGS` ở mục 1.
7. Windows: chỉ dùng 1 GPU (DDP / NCCL của PyTorch không chạy trên Windows). Linux có nhiều GPU thì dùng tất cả (DDP).

Mọi thứ notebook tạo ra nằm trong thư mục repo và đã có trong `.gitignore`:

```
eomt/
├── btxrd_cache/      ảnh đã thu nhỏ (mục 3)
├── pretrained/       checkpoint COCO, tải một lần (mục 4)
└── eomt_results/     kết quả: baseline/, tuned/, comparison/
```


## 1. Cấu hình

In [ ]:
import os
import sys
from pathlib import Path

# --- Paths: everything inside the cloned repo (data cache, checkpoints, results are git-ignored) ---
REPO_DIR = Path.cwd().resolve()
if not (REPO_DIR / "main.py").exists():
    raise RuntimeError(f"Working directory {REPO_DIR} is not the eomt repo: open this notebook from the cloned eomt folder")
os.chdir(REPO_DIR)
PY = sys.executable  # the kernel's Python: every subprocess (pip, main.py, tools/) uses the same environment

SMOKE_TEST = False  # True = 1 epoch x 2 batches per configuration to check the pipeline (written to <results>/smoke_test)

# --- Data ---
# Already-split BTXRD copied inside or next to the repo: used if present (first match), else Kaggle
DATA_SOURCES = [d / name for d in (REPO_DIR, REPO_DIR.parent) for name in ("BTXRD_full_split", "BTXRD_full_split.zip")]
KAGGLE_DATASET = "nguyetanhmai/btxrd-full-split"  # token: ~/.kaggle/kaggle.json or KAGGLE_USERNAME / KAGGLE_KEY
DATA_DIR = REPO_DIR / "btxrd_cache"  # resized copy written by tools/prepare_btxrd.py

# --- Outputs ---
RESULTS_DIR = REPO_DIR / "eomt_results"
PRETRAINED_DIR = REPO_DIR / "pretrained"  # COCO checkpoint, downloaded once then reused offline

# --- Shared by both configurations ---
BASE_CONFIG = "configs/dinov2/btxrd/instance/eomt_base_640.yaml"  # EoMT-B (DINOv2)
COCO_CKPT_REPO = "tue-mps/coco_panoptic_eomt_base_640_2x"  # fine-tune from EoMT-B COCO panoptic (README)
ANNEAL_EPOCHS = 5  # mask annealing packed into the first 5 epochs (project: 40): masked attention off during epoch 5
NUM_WORKERS = min(4 if os.name == "nt" else 8, os.cpu_count() or 2)  # data loading processes per GPU (Windows: spawn, keep RAM low)
RESUME = True  # continue from last.ckpt if it exists

# --- The two configurations (table above). "model" / "data": overrides of BASE_CONFIG ---
CONFIGS = {
    "baseline": {
        "title": "Tham số gốc của dự án",
        "dir": RESULTS_DIR / "baseline",
        "run_name": "btxrd_instance_eomt_base_640",
        "img_size": 640,
        "num_queries": 200,
        "max_epochs": 40,  # as the project config
        "patience": 3,  # counted from the end of mask annealing (epoch 5)
        "warmup_epochs": None,  # project: [500, 1000] steps
        "tune_thresholds": False,  # tumor / malignant thresholds 0.5 / 0.5
        "model": {},
        "data": {},
    },
    "tuned": {
        "title": "Siêu tham số điều chỉnh theo dữ liệu",
        "dir": RESULTS_DIR / "tuned",
        "run_name": "btxrd_instance_eomt_base_640_tuned",
        "img_size": 640,  # as the project (896 px: ~2x slower per epoch)
        "num_queries": 50,  # at most 9 tumors per image; the first 50 COCO query embeddings are kept
        "max_epochs": 40,  # as the project config (also the length of the poly lr schedule)
        "patience": 3,  # counted from the end of mask annealing (epoch 5)
        "warmup_epochs": (1, 2),  # heads alone, then backbone ramp-up
        "tune_thresholds": True,  # thresholds chosen on val (Youden J)
        "model": {
            "lr": 5e-5,  # effective batch 8 = half the batch 16 of the EoMT recipe; small dataset
            "llrd": 0.9,  # X-ray domain: early ViT blocks adapt more
            "weight_decay": 0.1,  # same decoupled decay lr x wd as the project
            "mask_coefficient": 2.0,
            "dice_coefficient": 5.0,
            "class_coefficient": 4.0,
            "no_object_coefficient": 0.1,
        },
        "data": {"scale_range": [0.8, 1.6], "balanced_sampling": True},
    },
}
RUN_CONFIGS = ["baseline", "tuned"]  # configurations trained / evaluated, in this order

for name, cfg in CONFIGS.items():
    out = cfg["dir"] / "smoke_test" if SMOKE_TEST else cfg["dir"]
    cfg.update(
        out=out,
        ckpt_dir=out / "checkpoints",
        run_yaml=out / "run.yaml",  # overrides of BASE_CONFIG for this run
        thresh_yaml=out / "thresholds.yaml",  # tumor / malignant thresholds used on test
        test_yaml=out / "test.yaml",  # overrides for evaluation
        train_done=out / "train_done.txt",  # written when training ends (early stop or max_epochs)
        best_weights=out / "best.bin",
        results=out / "results",
        val_results=out / "results" / "val",
        test_results=out / "results" / "test",
    )
    if name in RUN_CONFIGS:
        out.mkdir(parents=True, exist_ok=True)
        print(f"{name}: results -> {out}")
COMPARE_DIR = RESULTS_DIR / "comparison" / ("smoke_test" if SMOKE_TEST else "")
COMPARE_DIR.mkdir(parents=True, exist_ok=True)

## 2. Cài đặt thư viện

Mã nguồn dự án là thư mục repo đã clone. Cell dưới cài vào đúng môi trường của kernel (`sys.executable`):
1. PyTorch bản CUDA (`torch==2.7.0`, `torchvision==0.22.0`, như `requirements.txt`), chỉ khi môi trường chưa có PyTorch nhận được GPU.
2. `requirements_btxrd.txt` (giống notebook Colab), kèm `pycocotools==2.0.11` (bản 2.0.8 không có wheel cho Python 3.13 và build lỗi với NumPy 2).

Môi trường đã cài xong thì đặt `INSTALL_PACKAGES = False` để bỏ qua.

In [ ]:
import subprocess

INSTALL_PACKAGES = True  # False once the environment is ready
TORCH_PACKAGES = ["torch==2.7.0", "torchvision==0.22.0"]  # as requirements.txt
TORCH_INDEX = "https://download.pytorch.org/whl/cu128"  # CUDA 12.8 build; .../cu126 or .../cu118 for older drivers


def torch_has_cuda():
    """Checked in a subprocess, so torch is not imported in this kernel before it is installed."""
    r = subprocess.run([PY, "-c", "import torch; print(torch.cuda.is_available())"], capture_output=True, text=True)
    return r.stdout.strip() == "True"


def pip(*args):
    """pip install into the kernel's environment; its output is shown in the notebook."""
    r = subprocess.run([PY, "-m", "pip", "install", *args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(r.stdout[-3000:])
    if r.returncode != 0:
        raise RuntimeError(f"pip install {' '.join(args)} failed, see the output above")


if INSTALL_PACKAGES:
    if torch_has_cuda():
        print("PyTorch with CUDA already installed")
    else:
        pip(*TORCH_PACKAGES, "--index-url", TORCH_INDEX)
    pip("-q", "-r", "requirements_btxrd.txt")
    pip("-q", "pycocotools==2.0.11")
    if "torch" in sys.modules:
        print("torch was imported before the installation: restart the kernel, then Run All again")

In [ ]:
import torch

print(torch.__version__, "| CUDA available:", torch.cuda.is_available())  # '+cpu' = CPU build
!nvidia-smi
sys.path.insert(0, str(REPO_DIR))  # project modules (its datasets/ before Hugging Face `datasets`)

NUM_GPUS = torch.cuda.device_count()
if os.name == "nt" and NUM_GPUS > 1:
    NUM_GPUS = 1  # DDP (NCCL) is not available on Windows
if not NUM_GPUS:
    print("WARNING: no CUDA GPU visible, training would run on the CPU (very slow)")
print("GPUs used:", [torch.cuda.get_device_name(i) for i in range(NUM_GPUS)])

## 3. Chuẩn bị dữ liệu

`tools/prepare_btxrd.py` tải dataset Kaggle `nguyetanhmai/btxrd-full-split` (token Kaggle: xem phần **Chạy** ở đầu notebook), hoặc dùng thư mục `BTXRD_full_split` / `BTXRD_full_split.zip` nếu có trong hoặc cạnh thư mục repo; giữ nguyên cách chia trong `split.csv`, và ghi một bản ảnh đã thu nhỏ (cạnh dài ≤ 1024 px, polygon co theo) vào `btxrd_cache/`. Hai cấu hình dùng chung bản này. Chạy lại sẽ bỏ qua nếu đã chuẩn bị xong.

In [ ]:
data_src = next((p for p in DATA_SOURCES if p.exists()), None)
source = f'--src "{data_src}"' if data_src else f"--kaggle-dataset {KAGGLE_DATASET}"
print("Data source:", data_src or f"Kaggle {KAGGLE_DATASET}")
!"{PY}" tools/prepare_btxrd.py {source} --out "{DATA_DIR}"
if _exit_code != 0:
    raise RuntimeError("tools/prepare_btxrd.py failed, see the log above")

## 4. Huấn luyện hai cấu hình với early stopping

In [ ]:
from training.btxrd_callbacks import EarlyStoppingFrom, ModelCheckpointFrom  # code lives in the repo

print("Callbacks loaded from", ModelCheckpointFrom.__module__)

In [ ]:
from training.mask_classification_btxrd import MaskClassificationBTXRD  # code lives in the repo

assert "val_mask_ap" in MaskClassificationBTXRD.__init__.__code__.co_varnames, \
    "Repo clone is outdated: delete REPO_DIR and rerun section 2"
print("MaskClassificationBTXRD loaded from", MaskClassificationBTXRD.__module__)

In [ ]:
import copy
import math
import shutil

import yaml
from huggingface_hub import hf_hub_download
from timm.layers import resample_abs_pos_embed

from datasets.btxrd import BTXRDDataset, load_image_types

base_cfg = yaml.safe_load(Path(BASE_CONFIG).read_text())
BATCH_SIZE = base_cfg["data"]["init_args"]["batch_size"]  # 8 in the project config
BASE_IMG_SIZE = base_cfg["data"]["init_args"]["img_size"][0]  # 640
base_net = base_cfg["model"]["init_args"]["network"]["init_args"]
BASE_NUM_Q = base_net["num_q"]  # 200
NUM_BLOCKS = base_net["num_blocks"]
PATCH_SIZE = base_net["encoder"].get("init_args", {}).get("patch_size", 16)  # models.vit.ViT default
MONITOR = base_cfg["trainer"]["callbacks"][0]["init_args"]["monitor"]  # metrics/val_score
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 2**30 if NUM_GPUS else 0
n_train = len(BTXRDDataset(str(DATA_DIR), "train", load_image_types(str(DATA_DIR))))


def coco_ckpt_for(img_size, num_q):
    """EoMT-B COCO panoptic weights (640 px, 200 queries) adapted to img_size and num_q:
    pos_embed resampled to the img_size grid (if it differs), query embeddings cut to the first num_q."""
    src = PRETRAINED_DIR / "coco_panoptic_eomt_base_640_2x.bin"  # downloaded once, reused offline
    if not src.exists():
        PRETRAINED_DIR.mkdir(parents=True, exist_ok=True)
        shutil.copy(hf_hub_download(COCO_CKPT_REPO, "pytorch_model.bin"), src)
    if img_size == 640 and num_q == 200:
        return src
    dst = PRETRAINED_DIR / f"coco_panoptic_eomt_base_{img_size}_q{num_q}.bin"
    if dst.exists():
        return dst
    state = torch.load(src, map_location="cpu", weights_only=True)
    state = state.get("state_dict", state)
    q_keys = [k for k in state if k.endswith("network.q.weight")]
    assert q_keys, "query embedding not found in the COCO checkpoint"
    for key in q_keys:
        print(f"{key}: {tuple(state[key].shape)} -> ", end="")
        state[key] = state[key][:num_q].clone()
        print(tuple(state[key].shape))
    for key in [k for k in state if k.endswith("pos_embed")]:
        pos = state[key]
        old = math.isqrt(pos.shape[1])
        prefix = pos.shape[1] - old * old  # tokens without a grid position (none for DINOv2 reg4 in timm)
        new = img_size // PATCH_SIZE
        if new == old:
            continue  # same grid as the COCO checkpoint (640 px): pos_embed used as is
        state[key] = resample_abs_pos_embed(pos, new_size=(new, new), old_size=(old, old), num_prefix_tokens=prefix)
        print(f"{key}: {tuple(pos.shape)} -> {tuple(state[key].shape)}")
    torch.save(state, dst)
    return dst


def write_run_config(cfg):
    """Write run.yaml of one configuration; sets cfg["select_from_epoch"]."""
    img_size, num_q = cfg["img_size"], cfg["num_queries"]

    # Same effective batch on any GPU: the largest micro-batch that fits x gradient accumulation.
    # Rough memory need: ~2.5 GB per image at 640 px, growing with the number of tokens (img_size^2);
    # keep 15% of the GPU memory free as a margin
    gb_per_image = 2.5 * (img_size / 640) ** 2
    micro = max(b for b in (8, 4, 2, 1) if b <= BATCH_SIZE and (b == 1 or b * gb_per_image <= 0.85 * gpu_mem_gb))
    accumulate = BATCH_SIZE // micro

    # Mask annealing: formula of MaskClassificationBTXRD.on_fit_start applied to the optimizer
    # steps of the first ANNEAL_EPOCHS epochs instead of all max_epochs
    batches_per_epoch = math.ceil(n_train / max(1, NUM_GPUS)) // micro  # train loader: drop_last=True
    steps_per_epoch = math.ceil(batches_per_epoch / accumulate)
    unit = steps_per_epoch * ANNEAL_EPOCHS / 6
    anneal_start = [round(unit + k * 3 * unit / max(1, NUM_BLOCKS - 1)) for k in range(NUM_BLOCKS)]
    anneal_end = [round(s + unit) for s in anneal_start]
    # First epoch (0-based) whose validation runs with masked attention fully off
    cfg["select_from_epoch"] = 0 if SMOKE_TEST else math.ceil(anneal_end[-1] / steps_per_epoch) - 1

    # The project's checkpoint callback and early stopping on the same criterion,
    # both counting only the epochs after mask annealing
    checkpoint_cb = copy.deepcopy(base_cfg["trainer"]["callbacks"][0])
    checkpoint_cb["class_path"] = "training.btxrd_callbacks.ModelCheckpointFrom"
    checkpoint_cb["init_args"].update(dirpath=str(cfg["ckpt_dir"]), start_epoch=cfg["select_from_epoch"])
    early_stop_cb = {
        "class_path": "training.btxrd_callbacks.EarlyStoppingFrom",
        "init_args": {
            "monitor": MONITOR,
            "mode": checkpoint_cb["init_args"]["mode"],  # max
            "patience": cfg["patience"],
            "start_epoch": cfg["select_from_epoch"],
            "verbose": True,
        },
    }

    model_args = {
        "ckpt_path": str(coco_ckpt_for(img_size, num_q)),
        "load_ckpt_class_head": False,  # 133 COCO classes -> 2 BTXRD classes
        "attn_mask_annealing_start_steps": anneal_start,
        "attn_mask_annealing_end_steps": anneal_end,
        "val_mask_ap": False,  # mask AP only in main.py test
        **cfg["model"],
    }
    warmup = "project default"
    if cfg["warmup_epochs"]:
        # WARMUP_EPOCHS[0] epochs for the heads (backbone frozen), then the backbone ramps up
        # over WARMUP_EPOCHS[1] epochs (TwoStageWarmupPolySchedule)
        model_args["warmup_steps"] = warmup = [round(e * steps_per_epoch) for e in cfg["warmup_epochs"]]
    if num_q != BASE_NUM_Q:
        model_args["network"] = {"init_args": {"num_q": num_q}}
        model_args["eval_top_k_instances"] = min(100, num_q * 2)  # mask AP: top-k of queries x 2 classes
    data_args = {"path": str(DATA_DIR), "batch_size": micro, "num_workers": NUM_WORKERS, **cfg["data"]}
    if img_size != BASE_IMG_SIZE:
        data_args["img_size"] = [img_size, img_size]  # linked to the model, network and encoder by main.py

    run_cfg = {
        "trainer": {
            "max_epochs": 1 if SMOKE_TEST else cfg["max_epochs"],
            "devices": max(1, NUM_GPUS),
            "accelerator": "gpu" if NUM_GPUS else "cpu",
            "strategy": "ddp" if NUM_GPUS > 1 else "auto",
            "accumulate_grad_batches": accumulate,
            "log_every_n_steps": 50,
            "logger": {
                "class_path": "lightning.pytorch.loggers.CSVLogger",
                "init_args": {"save_dir": str(cfg["out"] / "logs"), "name": cfg["run_name"]},
            },
            "callbacks": [checkpoint_cb, early_stop_cb],
        },
        "model": {"init_args": model_args},
        "data": {"init_args": data_args},
    }
    if not NUM_GPUS:
        run_cfg["trainer"]["precision"] = "32-true"
    if SMOKE_TEST:
        run_cfg["trainer"].update(limit_train_batches=2, limit_val_batches=2, num_sanity_val_steps=0)
    cfg["run_yaml"].write_text(yaml.safe_dump(run_cfg, sort_keys=False))
    print(f"GPU memory {gpu_mem_gb:.0f} GB, {img_size} px -> batch {micro} x accumulate {accumulate} = {BATCH_SIZE}")
    print(f"{n_train} train images, {steps_per_epoch} steps/epoch, warm-up {warmup} | mask annealing ends "
          f"{anneal_end} (epoch {cfg['select_from_epoch'] + 1}) -> best model + early stopping from epoch "
          f"{cfg['select_from_epoch'] + 1} on, max {cfg['max_epochs']} epochs")


for name in RUN_CONFIGS:
    cfg = CONFIGS[name]
    print(f"\n===== {name}: {cfg['title']} =====")
    write_run_config(cfg)
    if cfg["train_done"].exists():
        print("Training already finished:", cfg["train_done"].read_text())
        continue
    run_yaml, last_ckpt = cfg["run_yaml"], cfg["ckpt_dir"] / "last.ckpt"
    resume = f'--ckpt_path "{last_ckpt}"' if RESUME and not SMOKE_TEST and last_ckpt.exists() else ""
    !"{PY}" main.py fit -c "{BASE_CONFIG}" -c "{run_yaml}" --compile_disabled {resume}
    if _exit_code != 0:
        raise RuntimeError(f"main.py fit failed for {name}, see the log above")
    best_ckpts = sorted(p.name for p in cfg["ckpt_dir"].glob("best-*.ckpt"))
    cfg["train_done"].write_text(f"best checkpoint: {best_ckpts}\n")
    print("Training finished:", cfg["train_done"].read_text())

### Diễn biến tiêu chí trên tập val

Đọc log CSV của từng cấu hình (gộp các lần resume): `val_score` và các thành phần theo từng epoch (đánh số từ 1). Cột `phase`: `annealing` = masked attention còn bật một phần, không dùng để chọn mô hình; `selection` = chạy như lúc suy luận, dùng để chọn mô hình tốt nhất (tô xanh) và đếm patience. Biểu đồ cuối đặt `val_score` của hai cấu hình cạnh nhau.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, Markdown, display

cols = {
    MONITOR: "val_score",
    "metrics/val_det_auroc": "det_auroc",
    "metrics/val_dice": "dice",
    "metrics/val_cls_auroc": "cls_auroc",
    "metrics/val_img3_acc": "img3_acc",
}
fig, ax = plt.subplots(figsize=(8, 4))
for name in RUN_CONFIGS:
    cfg = CONFIGS[name]
    display(Markdown(f"#### `{name}`: {cfg['title']}"))
    logs = sorted((cfg["out"] / "logs" / cfg["run_name"]).glob("version_*/metrics.csv"), key=os.path.getmtime)
    if not logs:
        print("No training log yet.")
        continue
    hist = pd.concat([pd.read_csv(p) for p in logs])
    val_hist = hist.dropna(subset=[MONITOR]).groupby("epoch")[list(cols)].last().rename(columns=cols)
    val_hist.insert(0, "phase", ["annealing" if e < cfg["select_from_epoch"] else "selection" for e in val_hist.index])
    val_hist.index += 1  # 1-based epochs, as in the text
    selection = val_hist[val_hist.phase == "selection"]
    cfg["results"].mkdir(parents=True, exist_ok=True)
    val_hist.to_csv(cfg["results"] / "val_history.csv")
    styled = val_hist.style.format("{:.4f}", subset=list(cols.values()))
    if len(selection):
        best_epoch = int(selection.val_score.idxmax())
        cfg["best_val_score"] = float(selection.val_score.max())
        styled = styled.apply(lambda r: ["background-color: #c6efce" if r.name == best_epoch else "" for _ in r], axis=1)
    display(styled)
    if len(selection):
        print(
            f"Best epoch (chosen among epochs >= {cfg['select_from_epoch'] + 1}): {best_epoch} "
            f"(val_score {cfg['best_val_score']:.4f}) | last epoch: {int(val_hist.index.max())} | "
            f"epochs without improvement at the end: {int(val_hist.index.max()) - best_epoch} "
            f"(patience {cfg['patience']}, max {cfg['max_epochs']} epochs)"
        )
    else:
        print(f"Mask annealing not finished yet: model selection starts at epoch {cfg['select_from_epoch'] + 1}")
    ax.plot(val_hist.index, val_hist.val_score, marker=".", label=name)
ax.axvline(min(CONFIGS[n]["select_from_epoch"] for n in RUN_CONFIGS) + 0.5, color="gray", ls="--", lw=1)
ax.set(xlabel="epoch", ylabel="val_score", title="val_score theo epoch (bên phải đường đứt: pha chọn mô hình)")
ax.legend()
ax.grid(alpha=0.3)
fig.savefig(COMPARE_DIR / "val_score.png", dpi=120, bbox_inches="tight")
plt.show()

## 5. Xuất trọng số tốt nhất và chọn ngưỡng trên tập val

Chuyển checkpoint tốt nhất (theo `metrics/val_score` trên val) của mỗi cấu hình thành state dict `best.bin`, định dạng README dùng cho `--model.ckpt_path`.

**Ngưỡng** (ghi vào `thresholds.yaml` của từng cấu hình, dùng cho mọi bước đánh giá trên test):
- `baseline`: 0,5 / 0,5 như dự án gốc.
- `tuned`: mô hình tốt nhất dự đoán toàn bộ tập val bằng `tools/predict_btxrd.py`, rồi hai ngưỡng được chọn bằng Youden J (tối đa độ nhạy + độ đặc hiệu, tức tối đa balanced accuracy). `tumor_score_thresh`: ảnh có u nếu P(u) > ngưỡng, xét trên mọi ảnh val. `malignant_thresh`: u ác nếu P(ác | u) ≥ ngưỡng, xét trên các ảnh val có u. Balanced accuracy không phụ thuộc tỉ lệ lớp, nên lớp thiểu số (u ác) được coi trọng ngang lớp đa số. Tập test không tham gia vào việc chọn ngưỡng.

In [ ]:
for name in RUN_CONFIGS:
    ckpt_dir, best_weights = CONFIGS[name]["ckpt_dir"], CONFIGS[name]["best_weights"]
    !"{PY}" tools/export_weights.py --ckpt-dir "{ckpt_dir}" --out "{best_weights}"

In [ ]:
import numpy as np
from sklearn.metrics import balanced_accuracy_score, roc_curve


def youden_threshold(y_true, score, strict):
    """Threshold maximizing sensitivity + specificity; strict=True for a `score > t` rule."""
    fpr, tpr, thr = roc_curve(y_true, score)
    ok = np.isfinite(thr) & (thr > 0.01) & (thr < 0.99)
    best = thr[ok][np.argmax((tpr - fpr)[ok])]
    return float(best - 1e-6) if strict else float(best)  # roc_curve predicts positive for score >= t


def summarize(df, tumor_t, mal_t):
    """Image-level metrics of predictions.csv rows for given thresholds."""
    has_tumor, tumor_rows = df.label > 0, df[df.label > 0]
    det = df.p_tumor > tumor_t
    mal = tumor_rows.p_malignant >= mal_t
    img3 = np.where(det, 1 + (df.p_malignant >= mal_t), 0)
    is_mal = tumor_rows.label == 2
    return {
        "det_sensitivity": det[has_tumor].mean(), "det_specificity": (~det[~has_tumor]).mean(),
        "det_balanced_acc": balanced_accuracy_score(has_tumor, det),
        "cls_sensitivity (malignant)": mal[is_mal].mean(), "cls_specificity (benign)": (~mal[~is_mal]).mean(),
        "cls_balanced_acc": balanced_accuracy_score(is_mal, mal),
        "img3_acc": (img3 == df.label).mean(), "img3_balanced_acc": balanced_accuracy_score(df.label, img3),
    }


for name in RUN_CONFIGS:
    cfg = CONFIGS[name]
    tumor_t = mal_t = 0.5
    if cfg["tune_thresholds"]:
        run_yaml, best_weights, val_results = cfg["run_yaml"], cfg["best_weights"], cfg["val_results"]
        !"{PY}" tools/predict_btxrd.py -c "{BASE_CONFIG}" -c "{run_yaml}" --weights "{best_weights}" --data-path "{DATA_DIR}" --split val --out "{val_results}" --num-vis 0
        val_pred = pd.read_csv(val_results / "predictions.csv")
        val_tumor = val_pred[val_pred.label > 0]
        tumor_t = youden_threshold(val_pred.label > 0, val_pred.p_tumor, strict=True)
        mal_t = youden_threshold(val_tumor.label == 2, val_tumor.p_malignant, strict=False)
        display(Markdown(f"**`{name}`: ngưỡng chọn trên val:** u `{tumor_t:.3f}`, ác `{mal_t:.3f}`"))
        display(pd.DataFrame({
            "val, ngưỡng 0.5 / 0.5": summarize(val_pred, 0.5, 0.5),
            "val, ngưỡng đã chọn": summarize(val_pred, tumor_t, mal_t),
        }).style.format("{:.4f}"))
    cfg["thresholds"] = (tumor_t, mal_t)
    cfg["thresh_yaml"].write_text(yaml.safe_dump(
        {"model": {"init_args": {"tumor_score_thresh": tumor_t, "malignant_thresh": mal_t}}}, sort_keys=False
    ))
    print(f"{name}: tumor {tumor_t:.3f}, malignant {mal_t:.3f} -> {cfg['thresh_yaml']}")

## 6. Đánh giá trên tập test (CLI của dự án)

Như phần *Evaluating* của README: tắt masked attention khi suy luận và nạp trọng số tốt nhất qua `model.ckpt_path`, chạy trên 1 GPU, với ngưỡng trong `thresholds.yaml`. In ra mask AP và các chỉ số phát hiện / phân đoạn / lành-ác / 3 lớp; bảng chỉ số của mỗi cấu hình được chép vào `<thư mục kết quả>/results/test_cli_metrics.csv`, bảng đặt hai cấu hình cạnh nhau hiện ở cuối cell.

In [ ]:
import shutil

TEST_KEYS = ["score", "ap_all", "det_auroc", "det_sensitivity", "det_specificity", "dice", "iou",
             "cls_auroc", "cls_balanced_acc", "img3_acc", "img3_balanced_acc"]
test_cli = {}
for name in RUN_CONFIGS:
    cfg = CONFIGS[name]
    test_cfg = {
        "trainer": {
            "devices": 1,
            "strategy": "auto",
            "logger": {
                "class_path": "lightning.pytorch.loggers.CSVLogger",
                "init_args": {"save_dir": str(cfg["out"] / "logs"), "name": f"{cfg['run_name']}_test"},
            },
        },
        "model": {
            "init_args": {
                "ckpt_path": str(cfg["best_weights"]),
                "load_ckpt_class_head": True,
                "network": {"init_args": {"masked_attn_enabled": False}},
            }
        },
    }
    cfg["test_yaml"].write_text(yaml.safe_dump(test_cfg, sort_keys=False))
    run_yaml, thresh_yaml, test_yaml = cfg["run_yaml"], cfg["thresh_yaml"], cfg["test_yaml"]
    print(f"\n===== {name} =====")
    !"{PY}" main.py test -c "{BASE_CONFIG}" -c "{run_yaml}" -c "{thresh_yaml}" -c "{test_yaml}"

    test_logs = sorted((cfg["out"] / "logs" / f"{cfg['run_name']}_test").glob("version_*/metrics.csv"),
                       key=os.path.getmtime)
    cfg["results"].mkdir(parents=True, exist_ok=True)
    shutil.copy(test_logs[-1], cfg["results"] / "test_cli_metrics.csv")
    row = pd.read_csv(test_logs[-1]).ffill().iloc[-1]
    test_cli[name] = {k: row.get(f"metrics/test_{k}", np.nan) for k in TEST_KEYS}

display(Markdown("**Chỉ số test từ `main.py test`** (`ap_all` = mask AP)"))
display(pd.DataFrame(test_cli).style.format("{:.4f}"))

## 7. Inference và đánh giá theo từng ảnh trên tập test

`tools/predict_btxrd.py` dự đoán từng ảnh (có u hay không, vùng u, lành / ác) với ngưỡng trong `thresholds.yaml`, ghi vào `<thư mục kết quả>/results/test/`: `predictions.csv`, báo cáo phân loại cho từng phần (`report.txt`, `metrics.json`), confusion matrix, ROC và hình minh họa `vis/`. Riêng phần lành / ác của `report.txt` / `metrics.json`, script luôn tính ở ngưỡng 0,5, nên bảng thứ hai tính lại với ngưỡng của từng cấu hình (`metrics_tuned_thresholds.csv`; với `baseline` hai bảng trùng nhau).

In [ ]:
import json

test_json, test_summary = {}, {}
for name in RUN_CONFIGS:
    cfg = CONFIGS[name]
    run_yaml, thresh_yaml, best_weights, test_results = (
        cfg["run_yaml"], cfg["thresh_yaml"], cfg["best_weights"], cfg["test_results"])
    print(f"\n===== {name} =====")
    !"{PY}" tools/predict_btxrd.py -c "{BASE_CONFIG}" -c "{run_yaml}" -c "{thresh_yaml}" --weights "{best_weights}" --data-path "{DATA_DIR}" --split test --out "{test_results}"
    test_json[name] = json.loads((test_results / "metrics.json").read_text())
    summary = summarize(pd.read_csv(test_results / "predictions.csv"), *cfg["thresholds"])
    pd.Series(summary, name="test").to_frame().to_csv(test_results / "metrics_tuned_thresholds.csv")
    test_summary[name] = summary

display(Markdown("**Chỉ số trên tập test (theo từng ảnh, `metrics.json`)**"))
display(pd.DataFrame(test_json))
display(Markdown("**Chỉ số trên tập test với ngưỡng của từng cấu hình**"))
display(pd.DataFrame(test_summary).style.format("{:.4f}"))
for name in RUN_CONFIGS:
    test_results = CONFIGS[name]["test_results"]
    display(Markdown(f"#### `{name}`: confusion matrix, ROC, vài ảnh test ngẫu nhiên"))
    display(Image(str(test_results / "confusion_matrix.png")), Image(str(test_results / "roc.png")))
    for p in sorted((test_results / "vis").glob("random_*.png"))[:3]:
        display(Image(str(p)))

## 8. So sánh hai cấu hình

Gộp `val_score` tốt nhất (mục 4), chỉ số `main.py test` (mục 6) và chỉ số theo từng ảnh với ngưỡng của từng cấu hình (mục 7) vào một bảng, cột `Δ (tuned − baseline)` tô xanh khi `tuned` tốt hơn. Lưu ở `eomt_results/comparison/comparison.csv`. Tập test chỉ có 187 ảnh (17 u ác), nên chênh lệch vài điểm phần trăm, nhất là ở phần lành / ác, có thể chỉ do ngẫu nhiên.

In [ ]:
comparison = pd.concat({
    "val": pd.DataFrame({n: {"best val_score": CONFIGS[n].get("best_val_score", np.nan)} for n in RUN_CONFIGS}),
    "test (main.py test)": pd.DataFrame(test_cli),
    "test (theo ảnh, ngưỡng của cấu hình)": pd.DataFrame(test_summary),
}).astype(float)
styled = comparison.style.format("{:.4f}")
if {"baseline", "tuned"} <= set(RUN_CONFIGS):
    comparison["Δ (tuned − baseline)"] = comparison.tuned - comparison.baseline
    styled = comparison.style.format("{:+.4f}", subset=["Δ (tuned − baseline)"]).format(
        "{:.4f}", subset=RUN_CONFIGS
    ).map(lambda v: "background-color: #c6efce" if v > 0 else "", subset=["Δ (tuned − baseline)"])
comparison.to_csv(COMPARE_DIR / "comparison.csv")
display(styled)

## 9. Một số kết quả trên tập test: dự đoán đúng và các trường hợp thất bại

Với **từng cấu hình**, chọn ca tiêu biểu cho từng nhóm từ `predictions.csv` của cấu hình đó, chạy lại mô hình tốt nhất trên các ảnh đó (hàm `build_model`, `predict`, `save_figure` của `tools/predict_btxrd.py`) và hiển thị: ảnh | ground truth | dự đoán (xanh lá = lành, đỏ = ác, viền vàng = vùng u thật). Bảng đầu tiên đặt số ca của từng nhóm ở hai cấu hình cạnh nhau.

| Nhóm | Điều kiện | Ca được chọn |
|---|---|---|
| Đúng: bình thường | bình thường → bình thường | `P(tumor)` thấp nhất |
| Đúng: u lành / u ác | đúng loại u | Dice cao nhất |
| Thất bại: u bị bỏ sót | có u → bình thường | sai tự tin nhất: `P(tumor)` thấp nhất |
| Thất bại: báo động nhầm | bình thường → có u | `P(tumor)` cao nhất |
| Thất bại: nhầm lành / ác | phát hiện có u nhưng sai loại | `P(malignant)` xa nhãn thật nhất |
| Thất bại: phân đoạn kém | phát hiện có u nhưng Dice < 0.5 | Dice thấp nhất |

Danh sách đầy đủ của mỗi cấu hình ghi vào `<thư mục kết quả>/results/test/examples.csv` (cột `group`, `failure`), hình vào `results/test/examples/`; bảng số ca ghi vào `eomt_results/comparison/example_counts.csv`.

In [ ]:
sys.path.insert(0, str(REPO_DIR / "tools"))
from predict_btxrd import build_model, merge, predict, save_figure  # noqa: E402

N_CORRECT = 2  # correct predictions shown per group
N_FAILURE = 3  # failure cases shown per type
LOW_DICE = 0.5


def example_groups(df):
    """key: (title, is failure, rows, sort key, ascending, out of)"""
    tumor, detected = df.label > 0, df.pred > 0
    wrong_type_conf = (df.p_malignant - (df.label == 2)).abs()  # distance of P(malignant) to the true label
    return {
        "correct_normal": ("Đúng: bình thường", False, (df.label == 0) & (df.pred == 0), df.p_tumor, True,
                           (df.label == 0).sum()),
        "correct_benign": ("Đúng: u lành", False, (df.label == 1) & (df.pred == 1), df.dice, False,
                           (df.label == 1).sum()),
        "correct_malignant": ("Đúng: u ác", False, (df.label == 2) & (df.pred == 2), df.dice, False,
                              (df.label == 2).sum()),
        "missed": ("Thất bại: u bị bỏ sót (có u → bình thường)", True, tumor & ~detected, df.p_tumor, True,
                   tumor.sum()),
        "false_alarm": ("Thất bại: báo động nhầm (bình thường → có u)", True, ~tumor & detected, df.p_tumor, False,
                        (~tumor).sum()),
        "swapped": ("Thất bại: nhầm lành / ác", True, tumor & detected & (df.label != df.pred), wrong_type_conf,
                    False, (tumor & detected).sum()),
        "low_dice": (f"Thất bại: phân đoạn kém (phát hiện có u, Dice < {LOW_DICE})", True,
                     tumor & detected & (df.dice < LOW_DICE), df.dice, True, (tumor & detected).sum()),
    }


examples_by, counts = {}, {}
for name in RUN_CONFIGS:
    df = pd.read_csv(CONFIGS[name]["test_results"] / "predictions.csv")
    groups = example_groups(df)
    examples = pd.concat(
        df[rows].assign(group=key, failure=failure, sort_key=sort_key[rows]).sort_values("sort_key", ascending=asc)
        for key, (title, failure, rows, sort_key, asc, _) in groups.items()
    )
    examples.to_csv(CONFIGS[name]["test_results"] / "examples.csv", index=False)
    examples_by[name] = (groups, examples)
    counts[name] = {key: f"{int(g[2].sum())} / {int(g[5])}" for key, g in groups.items()}

count_table = pd.DataFrame(counts)
count_table.insert(0, "group", [g[0] for g in groups.values()])
count_table.to_csv(COMPARE_DIR / "example_counts.csv")
display(Markdown("**Số ca theo nhóm** (số ca / số ca có thể rơi vào nhóm)"))
display(count_table)

device = torch.device("cuda" if NUM_GPUS else "cpu")
ds = BTXRDDataset(str(DATA_DIR), "test", load_image_types(str(DATA_DIR)))
index_of = {sample[0].stem: i for i, sample in enumerate(ds.samples)}
show_cols = ["image", "label_name", "pred_name", "p_tumor", "p_malignant", "dice"]
fmt = {"p_tumor": "{:.3f}", "p_malignant": "{:.3f}", "dice": "{:.3f}"}

for name in RUN_CONFIGS:
    cfg = CONFIGS[name]
    groups, examples = examples_by[name]
    display(Markdown(f"## `{name}`: {cfg['title']}"))

    # Re-run the best model on the cases shown (same read-out as predict_btxrd.py)
    config = {}
    for path in (BASE_CONFIG, cfg["run_yaml"], cfg["thresh_yaml"]):
        config = merge(config, yaml.safe_load(Path(path).read_text()))
    model = build_model(config, str(cfg["best_weights"]), device)
    fig_dir = cfg["test_results"] / "examples"
    fig_dir.mkdir(parents=True, exist_ok=True)

    for key, (title, failure, rows_mask, *_rest) in groups.items():
        rows = examples[examples.group == key].head(N_FAILURE if failure else N_CORRECT)
        display(Markdown(f"### `{name}` · {title}: {int(rows_mask.sum())} ca"))
        if rows.empty:
            print("Không có ca nào.")
            continue
        display(rows[show_cols].style.format(fmt, na_rep="-").hide(axis="index"))
        for _, row in rows.iterrows():
            img, target = ds[index_of[row.image]]
            pred_mask, *_ = predict(model, img, device)
            fig_path = fig_dir / f"{key}_{row.image}.png"
            save_figure(fig_path, img, target["masks"].bool().any(0), int(target["image_label"]), pred_mask, row)
            display(Image(str(fig_path)))

    del model
    torch.cuda.empty_cache()

## 10. Dự đoán cho ảnh X-quang mới

Mỗi cấu hình dự đoán cùng một bộ ảnh. Thay `NEW_IMAGES` bằng đường dẫn ảnh (hoặc thư mục ảnh) của bạn.

In [ ]:
NEW_IMAGES = sorted((DATA_DIR / "test" / "images").iterdir())[:3]  # example: 3 test images
image_args = " ".join(f'"{p}"' for p in NEW_IMAGES)
for name in RUN_CONFIGS:
    cfg = CONFIGS[name]
    run_yaml, thresh_yaml, best_weights = cfg["run_yaml"], cfg["thresh_yaml"], cfg["best_weights"]
    new_results = cfg["results"] / "new"
    display(Markdown(f"#### `{name}`: {cfg['title']}"))
    !"{PY}" tools/predict_btxrd.py -c "{BASE_CONFIG}" -c "{run_yaml}" -c "{thresh_yaml}" --weights "{best_weights}" --images {image_args} --out "{new_results}"
    for p in sorted((new_results / "vis").glob("*.png")):
        display(Image(str(p)))

## Kết quả đầu ra

Mỗi cấu hình có một thư mục kết quả: `baseline` → `eomt_results/baseline/`, `tuned` → `eomt_results/tuned/`.

| File (trong thư mục kết quả) | Nội dung |
|---|---|
| `best.bin` | Trọng số tốt nhất theo `val_score` (state dict), dùng với `--model.ckpt_path` |
| `checkpoints/best-epochXX.ckpt`, `last.ckpt` | Checkpoint Lightning (tốt nhất + cuối, dùng để resume) |
| `run.yaml`, `test.yaml` | Phần ghi đè cấu hình của lần chạy / lần đánh giá |
| `thresholds.yaml` | Ngưỡng u / ác dùng cho đánh giá trên test (`baseline`: 0,5 / 0,5; `tuned`: chọn trên val) |
| `train_done.txt` | Đánh dấu huấn luyện đã xong (xóa file này để huấn luyện lại) |
| `logs/` | Log CSV của huấn luyện và đánh giá (validation không có mask AP; mask AP chỉ có ở log test) |
| `results/val_history.csv` | `val_score` và các thành phần theo epoch (pha annealing / chọn mô hình) |
| `results/val/` | Dự đoán trên val dùng để chọn ngưỡng (chỉ `tuned`) |
| `results/test_cli_metrics.csv` | Chỉ số test từ `main.py test` |
| `results/test/` | `predictions.csv`, `report.txt`, `metrics.json`, `metrics_tuned_thresholds.csv`, `confusion_matrix.png`, `roc.png`, `vis/` |
| `results/test/examples.csv`, `results/test/examples/` | Ca dự đoán đúng tiêu biểu + các ca thất bại, kèm hình |
| `results/new/` | Dự đoán cho ảnh mới |

Chung cho hai cấu hình, trong `eomt_results/comparison/`: `comparison.csv` (bảng so sánh mục 8), `example_counts.csv` (số ca đúng / thất bại theo nhóm, mục 9) và `val_score.png` (biểu đồ mục 4).